<a href="https://colab.research.google.com/github/LevramUzo/gracce-groundwater-nigeria/blob/main/notebooks/01_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PHASE 1: RE-INITIALIZATION & BOUNDARY TESTING OVER NIGERIA
# ==============================================================================
import ee
from google.colab import userdata

try:
    # 1. Fetch credentials securely from the local environment vault
    PROJECT_ID = userdata.get('GEE_PROJECT_ID')

    # 2. Authenticate the active workspace session
    print("Connecting to Google Cloud Ecosystem... Allow permission via pop-up.")
    ee.Authenticate()

    # 3. Mount the engine to your cloud profile
    ee.Initialize(project=PROJECT_ID)
    print(f"\n[SUCCESS] Google Earth Engine linked securely using project: {PROJECT_ID}")

    # 4. Load the Global International Boundaries database and isolate Nigeria
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria_boundary = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # 5. Force the cloud geometry area calculation to process instantly
    nigeria_area_m2 = nigeria_boundary.geometry().area(maxError=1)
    nigeria_area_km2 = nigeria_area_m2.divide(1e6).getInfo()

    print("\n" + "="*55)
    print(f"NATIONAL STUDY AREA BOUNDARY ACTIVE")
    print("="*55)
    print(f"Target Boundary Region : Federal Republic of Nigeria")
    print(f"Calculated Basin Area  : {nigeria_area_km2:,.2f} km²")
    print("="*55 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Setup failed at re-initialization.")
    print(f"Details: {e}")


Connecting to Google Cloud Ecosystem... Allow permission via pop-up.


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python



[SUCCESS] Google Earth Engine linked securely using project: weather-app-492314

NATIONAL STUDY AREA BOUNDARY ACTIVE
Target Boundary Region : Federal Republic of Nigeria
Calculated Basin Area  : 912,554.10 km²



In [ ]:
# ==============================================================================
# STEP 2.2: HYDROCLIMATIC PREDICTOR EXTRACTION (CHIRPS & MODIS LST)
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Reference our Nigeria boundary layer from the first cell
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # Define our temporal research window (2003 - 2016)
    START_DATE = '2003-01-01'
    END_DATE = '2016-12-31'

    print("Step 1/3: Accessing Climate Catalogs inside Earth Engine...")

    # 2. Ingest the CHIRPS High-Resolution Precipitation Dataset
    chirps_collection = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY") \
        .filterDate(START_DATE, END_DATE) \
        .select('precipitation')

    # 3. Ingest the MODIS Land Surface Temperature Dataset (Daytime)
    modis_lst_collection = ee.ImageCollection("MODIS/061/MOD11A1") \
        .filterDate(START_DATE, END_DATE) \
        .select('LST_Day_1km')

    # 4. Define an extraction function to process monthly blocks over Nigeria
    def extract_monthly_climate(year, month):
        # Establish month-specific date bounds
        start = ee.Date.fromYMD(year, month, 1)
        end = start.advance(1, 'month')

        # Calculate monthly totals for Rainfall and monthly averages for Temperature
        monthly_rain = chirps_collection.filterDate(start, end).sum()
        monthly_temp = modis_lst_collection.filterDate(start, end).mean()

        # Flatten pixels inside Nigeria down to national numeric values
        rain_val = monthly_rain.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=5500, maxPixels=1e9
        ).get('precipitation')

        temp_val = monthly_temp.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=nigeria.geometry(), scale=5500, maxPixels=1e9
        ).get('LST_Day_1km')

        # Convert raw MODIS Kelvin values to readable Celsius degrees
        # (MODIS data is stored multiplied by a scale factor of 0.02)
        celsius_val = ee.Number(temp_val).multiply(0.02).subtract(273.15)

        # Return a structured record containing calendar keys
        date_str = start.format('YYYY-MM')
        return ee.Feature(None, {
            'date': date_str,
            'Rainfall_mm': rain_val,
            'LST_Celsius': ee.Algorithms.If(temp_val, celsius_val, None)
        })

    # 5. Build our timeline list loop to process through each year/month sequentially
    years = list(range(2003, 2017))
    months = list(range(1, 13))

    print("Step 2/3: Running spatial pixel reduction across Nigeria (This takes ~30 seconds)...")
    results_list = []

    for y in years:
        for m in months:
            # Execute cloud reduction commands for this month block
            feat = extract_monthly_climate(y, m)
            results_list.append(feat)

    # Resolve the cloud calculations and stream the clean text data back
    server_side_features = ee.FeatureCollection(results_list).getInfo()

    # 6. Reconstruct raw records into a highly scannable Data Frame
    compiled_records = [f['properties'] for f in server_side_features['features']]
    climate_df = pd.DataFrame(compiled_records)
    climate_df = climate_df.dropna().sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA CLIMATE MATRIX COMPILED")
    print("="*65)
    print(climate_df.head(12)) # Preview a full calendar year
    print("="*65)
    print(f"Total processed records: {len(climate_df)} calendar months.")
    print("="*65 + "\n")

except Exception as e:
    print(f"\n[ERROR]: Climate feature extraction failed.")
    print(f"Details: {e}")


Step 1/3: Accessing Climate Catalogs inside Earth Engine...
Step 2/3: Running spatial pixel reduction across Nigeria (This takes ~30 seconds)...

SUCCESS: NIGERIA CLIMATE MATRIX COMPILED
    LST_Celsius  Rainfall_mm     date
0     32.862381     6.546161  2003-01
1     35.399531    11.000401  2003-02
2     35.070512    16.483187  2003-03
3     37.283007    74.405237  2003-04
4     37.878470   112.640571  2003-05
5     29.155469   188.931344  2003-06
6     29.091259   216.928714  2003-07
7     28.365416   253.560656  2003-08
8     29.168214   231.016700  2003-09
9     31.794775    96.118767  2003-10
10    32.622224    20.124457  2003-11
11    31.650024     2.803890  2003-12
Total processed records: 168 calendar months.



In [3]:
# ==============================================================================
# STEP 2.3: TARGET VARIABLE EXTRACTION (NASA GRACE TWSA OVER NIGERIA)
# ==============================================================================
import ee
import pandas as pd

try:
    # 1. Fetch Nigeria's border using the global administrative layer
    global_boundaries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    nigeria = global_boundaries.filter(ee.Filter.eq('country_na', 'Nigeria'))

    # Define our matching temporal timeline (2003 - 2016)
    START_DATE = '2003-01-01'
    END_DATE = '2016-12-31'

    print("Accessing NASA GRACE Satellite Gravity Grids...")

    # 2. Ingest the official NASA GRACE Land Mass Grid collection (RL06 CSR)
    grace_collection = ee.ImageCollection("NASA/GRACE/MASS_GRIDS_V04/LAND") \
        .filterDate(START_DATE, END_DATE) \
        .select('lwe_thickness_csr')

    # 3. Create a regional function to calculate average water weight anomalies
    def extract_monthly_grace(image):
        # Reduce global pixels down to a single average thickness number inside Nigeria
        mean_val = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=nigeria.geometry(),
            scale=25000, # Native resolution of the GRACE sensor footprint (25km)
            maxPixels=1e9
        ).get('lwe_thickness_csr')

        # Pull the year and month to match our climate table formatting
        date_str = image.date().format('YYYY-MM')

        return ee.Feature(None, {
            'date': date_str,
            'TWSA_mm': mean_val
        })

    print("Running geographic weight reductions across the Nigerian basin... Please wait.")

    # 4. Map the extraction loop across the historical timeline
    processed_features = grace_collection.map(extract_monthly_grace).getInfo()

    # 5. Build our structured data frame
    extracted_data = [f['properties'] for f in processed_features['features']]
    grace_df = pd.DataFrame(extracted_data)
    grace_df = grace_df.dropna().sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: NIGERIA GRACE TARGET ANOMALIES COMPILED")
    print("="*65)
    print(grace_df.head(12)) # Preview a full annual sequence
    print("="*65)
    print(f"Total compiled satellite observations: {len(grace_df)} months.")
    print("="*65 + "\n")

except Exception as e:
    print(f"\n[ERROR]: GRACE feature extraction failed.")
    print(f"Details: {e}")


Accessing NASA GRACE Satellite Gravity Grids...
Running geographic weight reductions across the Nigerian basin... Please wait.

SUCCESS: NIGERIA GRACE TARGET ANOMALIES COMPILED
     TWSA_mm     date
0  -0.091221  2003-01
1  -0.119114  2003-02
2  -0.133866  2003-03
3  -0.144336  2003-04
4  -0.013229  2003-06
5   0.055186  2003-07
6   0.144293  2003-08
7   0.133955  2003-09
8   0.070251  2003-10
9   0.017593  2003-11
10 -0.017884  2003-12
11 -0.061574  2004-02
Total compiled satellite observations: 150 months.



In [6]:
# ==============================================================================
# STEP 2.4: DATASET MERGING AND COORDINATION
# ==============================================================================
import pandas as pd

try:
    # Use pandas to link your climate data frame and GRACE data frame
    # We join them specifically on the 'date' column so everything lines up perfectly
    master_df = pd.merge(climate_df, grace_df, on='date', how='inner')

    # Sort the rows chronologically from past to present
    master_df = master_df.sort_values('date').reset_index(drop=True)

    print("\n" + "="*65)
    print("SUCCESS: MASTER HYDRO-MACHINE LEARNING DATASET MERGED")
    print("="*65)
    print(master_df.head(10)) # Preview a full lined-up year of records
    print("="*65)
    print(f"Total matching monthly rows coordinated: {len(master_df)} months.")
    print("="*65 + "\n")

    # Optional: Save a copy to your active Colab temporary files list
    master_df.to_csv('nigeria_groundwater_master.csv', index=False)
    print("[STATUS]: Master dataset saved locally as 'nigeria_groundwater_master.csv'")

except Exception as e:
    print(f"\n[ERROR]: Dataset merge failed.")
    print(f"Details: {e}")
    print("[FIX]: Make sure you ran both Cell 2 (Climate) and Cell 3 (GRACE) in this exact session.")



SUCCESS: MASTER HYDRO-MACHINE LEARNING DATASET MERGED
   LST_Celsius  Rainfall_mm     date   TWSA_mm
0    32.862381     6.546161  2003-01 -0.091221
1    35.399531    11.000401  2003-02 -0.119114
2    35.070512    16.483187  2003-03 -0.133866
3    37.283007    74.405237  2003-04 -0.144336
4    29.155469   188.931344  2003-06 -0.013229
5    29.091259   216.928714  2003-07  0.055186
6    28.365416   253.560656  2003-08  0.144293
7    29.168214   231.016700  2003-09  0.133955
8    31.794775    96.118767  2003-10  0.070251
9    32.622224    20.124457  2003-11  0.017593
Total matching monthly rows coordinated: 150 months.

[STATUS]: Master dataset saved locally as 'nigeria_groundwater_master.csv'
